In [22]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ langchain-community  — installed (0.4.2)
  ✓ lxml  — installed (6.1.1)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Fixed-size Chunking

Fixed-size chunking splits text into fixed-length windows (with a small overlap), ignoring document structure. It's the simplest baseline strategy.

Below we first **load** a couple of source documents (a PDF and an HTML page), then apply fixed-size chunking to them.

## 1. Load source documents

### PDF

Load the sample PDF with `PyPDFLoader` (one `Document` per page).

In [23]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
pdf_docs = PyPDFLoader(str(pdf_path)).load()

print(f"PDF: loaded {len(pdf_docs)} page(s)")
print(pdf_docs[0].page_content[:300])

PDF: loaded 12 page(s)
SDLC — End-to-End Reference
Page 1
 Software Development Life Cycle
 End-to-End Artifacts & Deliverables
From Business Requirements (BRD) to Release Notes and Operations
 A reference guide describing each SDLC document, its purpose, owner, inputs,
 key contents, and how it feeds the next stage.
 Aut


### HTML

Load the sample HTML with `BSHTMLLoader` (cleaned text in one `Document`).

In [24]:
from langchain_community.document_loaders import BSHTMLLoader

html_path = ROOT / "assets/RAG_Courses.html"
html_docs = BSHTMLLoader(str(html_path)).load()

print(f"HTML: loaded {len(html_docs)} document(s)")
print(html_docs[0].page_content[:300])

HTML: loaded 1 document(s)









12 Best Retrieval-Augmented Generation (RAG) Courses in 2026 — Class Central



















































The Four-Year Degree Isn't Dying — It's Evolving



			View
			




		Close
	









The Report by Class Central


Your source for the latest news and trends in 


## 2. Fixed-size chunking

In [25]:
from langchain_text_splitters import CharacterTextSplitter

# separator="" → no structural break points, so text is cut into pure
# fixed-length character windows with a fixed overlap between them.
splitter = CharacterTextSplitter(
    separator="",
    chunk_size=500,      # characters per chunk
    chunk_overlap=50,    # characters shared between consecutive chunks
)

### `split_text` vs `split_documents`

Both apply the **same** splitting logic; they differ only in input/output:

| Method | Input | Output | Metadata |
|--------|-------|--------|----------|
| `split_text(text: str)` | a raw string | `list[str]` (plain strings) | **lost** — no `Document` wrapper |
| `split_documents(docs: list[Document])` | `Document` objects (from a loader) | `list[Document]` (chunks) | **preserved** — each source's `.metadata` (page number, source path, …) is copied onto every chunk |

For RAG you usually want **`split_documents`**, so each chunk stays traceable to its origin document.

In [26]:
# Pass a raw string -> `chunks` is a list[str] with no metadata (see note above).
chunks = text_splitter.split_text(html_docs[0].page_content)
print(f"Split into {len(chunks)} chunks; first chunk is {len(chunks[0])} characters")

Created a chunk of size 77, which is longer than the specified 75
Created a chunk of size 85, which is longer than the specified 75
Created a chunk of size 125, which is longer than the specified 75
Created a chunk of size 80, which is longer than the specified 75
Created a chunk of size 145, which is longer than the specified 75
Created a chunk of size 1010, which is longer than the specified 75
Created a chunk of size 164, which is longer than the specified 75
Created a chunk of size 84, which is longer than the specified 75
Created a chunk of size 81, which is longer than the specified 75
Created a chunk of size 100, which is longer than the specified 75
Created a chunk of size 110, which is longer than the specified 75
Created a chunk of size 79, which is longer than the specified 75
Created a chunk of size 1575, which is longer than the specified 75
Created a chunk of size 377, which is longer than the specified 75
Created a chunk of size 2513, which is longer than the specified 7

Split into 247 chunks; first chunk is 76 characters


In [27]:
# Combine the loaded documents and split them into fixed-size chunks.
docs = pdf_docs + html_docs
chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} documents into {len(chunks)} fixed-size chunks")
print(f"First chunk ({len(chunks[0].page_content)} chars):\n{chunks[0].page_content[:300]}")

Split 13 documents into 126 fixed-size chunks
First chunk (361 chars):
SDLC — End-to-End Reference
Page 1
 Software Development Life Cycle
 End-to-End Artifacts & Deliverables
From Business Requirements (BRD) to Release Notes and Operations
 A reference guide describing each SDLC document, its purpose, owner, inputs,
 key contents, and how it feeds the next stage.
 Aut
